# K-Nearest Neighbors from Scratch

**Interview-focused reference notebook** -- Brute-force KNN, KD-Tree, distance metrics, curse of dimensionality.

## Core Intuition

KNN is a **lazy learner** -- it stores all training data and defers computation to prediction time. No explicit training phase.

**Algorithm:** For a query point $\mathbf{x}_q$:
1. Compute distance from $\mathbf{x}_q$ to every training point
2. Find $K$ nearest neighbors
3. Classification: majority vote. Regression: average.

**Complexity:**
- Training: $O(1)$ (just store the data)
- Prediction (brute-force): $O(nd)$ per query (n=samples, d=features)
- Prediction (KD-Tree): $O(d \log n)$ average case, $O(nd)$ worst case
- Space: $O(nd)$

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter
np.random.seed(42)
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 11

## 1. Distance Metrics

In [ ]:
def euclidean_distance(x1, x2):
    """L2 distance: sqrt(sum((x1-x2)^2))"""
    return np.sqrt(np.sum((x1 - x2) ** 2, axis=-1))

def manhattan_distance(x1, x2):
    """L1 distance: sum(|x1-x2|)"""
    return np.sum(np.abs(x1 - x2), axis=-1)

def cosine_distance(x1, x2):
    """1 - cosine_similarity. Good for high-dimensional sparse data (text)."""
    dot = np.sum(x1 * x2, axis=-1)
    norm1 = np.sqrt(np.sum(x1 ** 2, axis=-1))
    norm2 = np.sqrt(np.sum(x2 ** 2, axis=-1))
    return 1 - dot / (norm1 * norm2 + 1e-12)

def minkowski_distance(x1, x2, p=2):
    """Generalized Lp distance. p=1 is Manhattan, p=2 is Euclidean."""
    return np.sum(np.abs(x1 - x2) ** p, axis=-1) ** (1.0 / p)

# Visualize distance metric "circles" (level sets)
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
theta = np.linspace(0, 2*np.pi, 500)
origin = np.array([0.0, 0.0])

for ax, p, name in [(axes[0], 1, 'Manhattan (p=1)'),
                     (axes[1], 2, 'Euclidean (p=2)'),
                     (axes[2], 10, 'Minkowski (p=10)')]:
    # Points at unit distance from origin
    grid_x = np.linspace(-1.5, 1.5, 200)
    grid_y = np.linspace(-1.5, 1.5, 200)
    GX, GY = np.meshgrid(grid_x, grid_y)
    D = (np.abs(GX)**p + np.abs(GY)**p)**(1/p)
    ax.contour(GX, GY, D, levels=[1.0], colors='blue', linewidths=2)
    ax.set_aspect('equal')
    ax.set_title(f'{name}')
    ax.grid(True, alpha=0.3)
    ax.set_xlim(-1.5, 1.5)
    ax.set_ylim(-1.5, 1.5)
    ax.axhline(y=0, color='k', alpha=0.2)
    ax.axvline(x=0, color='k', alpha=0.2)

plt.suptitle('Unit "Circles" for Different Metrics', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

## 2. Synthetic Data

In [ ]:
from sklearn.datasets import make_moons, make_circles, make_classification

X_moons, y_moons = make_moons(n_samples=300, noise=0.2, random_state=42)
X_circles, y_circles = make_circles(n_samples=300, noise=0.1, factor=0.5, random_state=42)
X_blobs, y_blobs = make_classification(n_samples=300, n_features=2, n_redundant=0,
                                        n_informative=2, n_clusters_per_class=1, random_state=42)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, X, y, title in [(axes[0], X_moons, y_moons, 'Moons'),
                          (axes[1], X_circles, y_circles, 'Circles'),
                          (axes[2], X_blobs, y_blobs, 'Blobs')]:
    ax.scatter(X[:, 0], X[:, 1], c=y, cmap='RdBu', s=30, alpha=0.7, edgecolors='k')
    ax.set_title(title)
plt.tight_layout()
plt.show()
print("KNN excels on moons/circles because it creates local, nonlinear decision boundaries.")

## 3. Brute-Force KNN Implementation

In [ ]:
class KNNClassifier:
    """K-Nearest Neighbors classifier (brute-force)."""
    
    def __init__(self, k=5, distance='euclidean', weighted=False):
        self.k = k
        self.weighted = weighted
        self.distance_func = {
            'euclidean': euclidean_distance,
            'manhattan': manhattan_distance,
            'cosine': cosine_distance
        }[distance]
    
    def fit(self, X, y):
        """Just store the data -- lazy learning."""
        self.X_train = X.copy()
        self.y_train = y.copy()
        self.classes_ = np.unique(y)
        return self
    
    def _predict_single(self, x):
        """Predict class for a single query point."""
        # Compute distances to all training points
        distances = self.distance_func(self.X_train, x)
        
        # Find K nearest neighbors
        k_indices = np.argsort(distances)[:self.k]
        k_labels = self.y_train[k_indices]
        k_distances = distances[k_indices]
        
        if self.weighted:
            # Distance-weighted voting: closer neighbors get more weight
            weights = 1.0 / (k_distances + 1e-12)
            class_weights = {}
            for label, weight in zip(k_labels, weights):
                class_weights[label] = class_weights.get(label, 0) + weight
            return max(class_weights, key=class_weights.get)
        else:
            # Simple majority vote
            return Counter(k_labels).most_common(1)[0][0]
    
    def predict(self, X):
        return np.array([self._predict_single(x) for x in X])


class KNNRegressor:
    """K-Nearest Neighbors regressor (brute-force)."""
    
    def __init__(self, k=5, weighted=False):
        self.k = k
        self.weighted = weighted
    
    def fit(self, X, y):
        self.X_train = X.copy()
        self.y_train = y.copy()
        return self
    
    def _predict_single(self, x):
        distances = euclidean_distance(self.X_train, x)
        k_indices = np.argsort(distances)[:self.k]
        k_values = self.y_train[k_indices]
        k_distances = distances[k_indices]
        
        if self.weighted:
            weights = 1.0 / (k_distances + 1e-12)
            return np.average(k_values, weights=weights)
        else:
            return np.mean(k_values)
    
    def predict(self, X):
        return np.array([self._predict_single(x) for x in X])

# Test on moons data
knn = KNNClassifier(k=5).fit(X_moons, y_moons)
acc = np.mean(knn.predict(X_moons) == y_moons)
print(f"KNN (k=5) accuracy on moons: {acc:.4f}")

knn_w = KNNClassifier(k=5, weighted=True).fit(X_moons, y_moons)
acc_w = np.mean(knn_w.predict(X_moons) == y_moons)
print(f"Weighted KNN (k=5) accuracy on moons: {acc_w:.4f}")

## 4. Decision Boundaries for Different K Values

In [ ]:
def plot_knn_boundary(X, y, k, ax, title=''):
    model = KNNClassifier(k=k).fit(X, y)
    h = 0.05
    x_min, x_max = X[:, 0].min() - 0.5, X[:, 0].max() + 0.5
    y_min, y_max = X[:, 1].min() - 0.5, X[:, 1].max() + 0.5
    xx, yy = np.meshgrid(np.arange(x_min, x_max, h),
                          np.arange(y_min, y_max, h))
    Z = model.predict(np.column_stack([xx.ravel(), yy.ravel()]))
    Z = Z.reshape(xx.shape)
    
    ax.contourf(xx, yy, Z, alpha=0.3, cmap='RdBu')
    ax.scatter(X[:, 0], X[:, 1], c=y, cmap='RdBu', s=20, alpha=0.7, edgecolors='k', linewidths=0.5)
    ax.set_title(title)

fig, axes = plt.subplots(2, 3, figsize=(15, 9))
k_values = [1, 3, 5, 15, 30, 100]

for ax, k in zip(axes.ravel(), k_values):
    plot_knn_boundary(X_moons, y_moons, k, ax, f'K = {k}')

plt.suptitle('KNN Decision Boundaries (Moons Dataset)', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()
print("K=1: overfitting (jagged boundary, memorizes noise).")
print("K=100: underfitting (too smooth, ignores local structure).")
print("K=5-15: good balance for this dataset.")

## 5. KD-Tree for Efficient Nearest Neighbor Search

A KD-Tree partitions space using axis-aligned splits (alternating dimensions). Average query time: $O(d \log n)$.

**Construction:** At each level, pick the dimension with the most spread, split at the median.

**Search:** Branch-and-bound -- prune subtrees that can't contain closer points.

**Limitation:** Degrades to $O(nd)$ in high dimensions (curse of dimensionality).

In [ ]:
class KDNode:
    """A node in the KD-Tree."""
    def __init__(self, point, label, axis, left=None, right=None):
        self.point = point
        self.label = label
        self.axis = axis
        self.left = left
        self.right = right

class KDTree:
    """KD-Tree for efficient nearest neighbor search."""
    
    def __init__(self):
        self.root = None
    
    def build(self, X, y, depth=0):
        """Recursively build the KD-Tree."""
        if len(X) == 0:
            return None
        
        d = X.shape[1]
        axis = depth % d  # alternate splitting dimensions
        
        # Sort by the current axis and pick the median
        sorted_indices = np.argsort(X[:, axis])
        median_idx = len(sorted_indices) // 2
        
        node = KDNode(
            point=X[sorted_indices[median_idx]],
            label=y[sorted_indices[median_idx]],
            axis=axis,
            left=self.build(X[sorted_indices[:median_idx]], y[sorted_indices[:median_idx]], depth + 1),
            right=self.build(X[sorted_indices[median_idx+1:]], y[sorted_indices[median_idx+1:]], depth + 1)
        )
        return node
    
    def fit(self, X, y):
        self.root = self.build(X, y)
        return self
    
    def _search(self, node, query, k, heap):
        """Recursively search for k nearest neighbors using branch-and-bound."""
        if node is None:
            return
        
        dist = np.sqrt(np.sum((node.point - query) ** 2))
        
        # heap stores (negative_distance, point, label) -- max-heap on distance
        if len(heap) < k:
            heap.append((dist, node.point.copy(), node.label))
            heap.sort(key=lambda x: -x[0])  # keep sorted, largest first
        elif dist < heap[0][0]:
            heap[0] = (dist, node.point.copy(), node.label)
            heap.sort(key=lambda x: -x[0])
        
        # Determine which subtree to search first
        diff = query[node.axis] - node.point[node.axis]
        if diff <= 0:
            first, second = node.left, node.right
        else:
            first, second = node.right, node.left
        
        # Search the closer subtree first
        self._search(first, query, k, heap)
        
        # Only search the other subtree if it could contain closer points
        if len(heap) < k or abs(diff) < heap[0][0]:
            self._search(second, query, k, heap)
    
    def query(self, point, k=1):
        """Find k nearest neighbors."""
        heap = []
        self._search(self.root, point, k, heap)
        heap.sort(key=lambda x: x[0])  # sort by distance ascending
        return heap


class KNNClassifierKDTree:
    """KNN classifier using KD-Tree."""
    
    def __init__(self, k=5):
        self.k = k
    
    def fit(self, X, y):
        self.tree = KDTree().fit(X, y)
        return self
    
    def predict(self, X):
        predictions = []
        for x in X:
            neighbors = self.tree.query(x, self.k)
            labels = [n[2] for n in neighbors]
            predictions.append(Counter(labels).most_common(1)[0][0])
        return np.array(predictions)

# Compare brute-force vs KD-Tree
knn_brute = KNNClassifier(k=5).fit(X_moons, y_moons)
knn_kd = KNNClassifierKDTree(k=5).fit(X_moons, y_moons)

pred_brute = knn_brute.predict(X_moons[:20])
pred_kd = knn_kd.predict(X_moons[:20])

print(f"Brute-force predictions: {pred_brute}")
print(f"KD-Tree predictions:     {pred_kd}")
print(f"Match: {np.all(pred_brute == pred_kd)}")

## 6. Speed Comparison: Brute-Force vs KD-Tree

In [ ]:
import time

# Generate larger dataset for timing
X_large = np.random.randn(2000, 2)
y_large = (X_large[:, 0] + X_large[:, 1] > 0).astype(int)
X_query = np.random.randn(200, 2)

# Brute-force timing
knn_b = KNNClassifier(k=5).fit(X_large, y_large)
start = time.time()
pred_b = knn_b.predict(X_query)
time_brute = time.time() - start

# KD-Tree timing (includes build time)
start = time.time()
knn_k = KNNClassifierKDTree(k=5).fit(X_large, y_large)
time_build = time.time() - start

start = time.time()
pred_k = knn_k.predict(X_query)
time_kdtree = time.time() - start

print(f"Brute-force query time: {time_brute:.4f}s")
print(f"KD-Tree build time:     {time_build:.4f}s")
print(f"KD-Tree query time:     {time_kdtree:.4f}s")
print(f"\nNote: KD-Tree pays a build cost but wins on repeated queries.")
print(f"For very high dimensions (d > ~20), KD-Tree loses its advantage.")

## 7. Curse of Dimensionality Demo

In high dimensions, all points become approximately equidistant. The concept of "nearest neighbor" loses meaning.

In [ ]:
# Demonstrate: ratio of max-to-min distance approaches 1 in high dimensions
dims = [2, 5, 10, 20, 50, 100, 200, 500, 1000]
n_points = 1000
n_trials = 10

ratios = []
nn_accs = []

for d in dims:
    trial_ratios = []
    trial_accs = []
    
    for _ in range(n_trials):
        X = np.random.randn(n_points, d)
        query = np.random.randn(1, d)
        
        # Distance concentration
        dists = np.sqrt(np.sum((X - query) ** 2, axis=1))
        ratio = (dists.max() - dists.min()) / dists.min()
        trial_ratios.append(ratio)
        
        # KNN accuracy with simple linear boundary
        y = (X[:, 0] > 0).astype(int)  # only first feature matters
        knn_model = KNNClassifier(k=5).fit(X[:800], y[:800])
        pred = knn_model.predict(X[800:900])  # small test set for speed
        trial_accs.append(np.mean(pred == y[800:900]))
    
    ratios.append(np.mean(trial_ratios))
    nn_accs.append(np.mean(trial_accs))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(dims, ratios, 'bo-', linewidth=2, markersize=8)
axes[0].set_xlabel('Number of Dimensions')
axes[0].set_ylabel('(max_dist - min_dist) / min_dist')
axes[0].set_title('Distance Concentration: Distances Become Similar')
axes[0].grid(True, alpha=0.3)
axes[0].set_xscale('log')

axes[1].plot(dims, nn_accs, 'ro-', linewidth=2, markersize=8)
axes[1].set_xlabel('Number of Dimensions')
axes[1].set_ylabel('KNN Accuracy (K=5)')
axes[1].set_title('KNN Accuracy Degrades with Dimensionality')
axes[1].grid(True, alpha=0.3)
axes[1].set_xscale('log')
axes[1].set_ylim(0.4, 1.05)

plt.tight_layout()
plt.show()
print("As d grows, relative distance differences shrink -> 'nearest' neighbor is barely closer than 'farthest'.")
print("This is why KNN struggles in high dimensions without dimensionality reduction.")

## 8. KNN Regression Demo

In [ ]:
# 1D regression example
X_reg = np.sort(np.random.uniform(0, 10, 50)).reshape(-1, 1)
y_reg = np.sin(X_reg.ravel()) + 0.3 * np.random.randn(50)
X_plot = np.linspace(0, 10, 300).reshape(-1, 1)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, k in zip(axes, [1, 5, 20]):
    knn_reg = KNNRegressor(k=k).fit(X_reg, y_reg)
    y_pred = knn_reg.predict(X_plot)
    
    ax.scatter(X_reg, y_reg, c='blue', s=30, alpha=0.7, label='Data')
    ax.plot(X_plot, y_pred, 'r-', linewidth=2, label=f'KNN (K={k})')
    ax.plot(X_plot, np.sin(X_plot), 'g--', alpha=0.5, label='True function')
    ax.set_title(f'KNN Regression (K={k})')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()
print("K=1: piecewise constant, noisy. K=20: smoother but misses details.")

## 9. Sklearn Comparison

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

datasets = [
    ('Moons', X_moons, y_moons),
    ('Circles', X_circles, y_circles),
    ('Blobs', X_blobs, y_blobs)
]

print(f"{'Dataset':<12} {'K':>3} {'Ours':>10} {'sklearn':>10}")
print("-" * 38)

for name, X, y in datasets:
    for k in [3, 5, 10]:
        our_model = KNNClassifier(k=k).fit(X, y)
        sk_model = KNeighborsClassifier(n_neighbors=k).fit(X, y)
        
        our_acc = accuracy_score(y, our_model.predict(X))
        sk_acc = sk_model.score(X, y)
        print(f"{name:<12} {k:>3} {our_acc:>10.4f} {sk_acc:>10.4f}")

## 10. Connection to Kernel Methods and LSH

**Kernel methods:** KNN with distance-weighted voting is equivalent to kernel regression with a particular kernel (the indicator function for unweighted, or an inverse-distance kernel for weighted).

**Locality Sensitive Hashing (LSH):** For very large datasets where even KD-Tree is slow, LSH provides approximate nearest neighbors in sublinear time by hashing similar points to the same bucket.

| Method | Build Time | Query Time | Exact? |
|--------|-----------|------------|--------|
| Brute-force | $O(1)$ | $O(nd)$ | Yes |
| KD-Tree | $O(nd \log n)$ | $O(d \log n)$ avg | Yes |
| Ball Tree | $O(nd \log n)$ | $O(d \log n)$ avg | Yes |
| LSH | $O(n)$ | $O(1)$ avg | No (approximate) |
| HNSW/FAISS | $O(n \log n)$ | $O(\log n)$ avg | No (approximate) |

## Interview Quick Reference

| Question | Answer |
|----------|--------|
| Time complexity of KNN? | Training: $O(1)$. Prediction: $O(nd)$ brute-force, $O(d \log n)$ with KD-Tree. |
| How to handle high dimensions? | Dimensionality reduction (PCA), feature selection, or use approximate methods (LSH, HNSW). KD-Tree degrades past ~20 dimensions. |
| KNN vs radius-based neighbors? | KNN: fixed K, variable radius. Radius NN: fixed radius, variable neighbors. Radius NN struggles with varying density. |
| How to choose K? | Cross-validation. Small K = low bias, high variance. Large K = high bias, low variance. Odd K avoids ties in binary classification. |
| Does KNN need feature scaling? | Yes! Distance-based methods are sensitive to feature scales. Standardize or normalize first. |
| Weighted vs unweighted? | Weighted (1/distance) reduces sensitivity to K and gives smoother boundaries. |
| When does KNN shine? | Low dimensions, nonlinear boundaries, enough data. Good for moons, circles, complex local structure. |